# Abacus Pasting Optimization Audit - HEALPix Ring Backend

Created UTC: `2026-06-08T00:51:22Z`

This notebook records the post-audit status of the recommended optimization work in `summary/ABACUS_BACKLIGHT_PASTING_OPTIMIZATION_SUMMARY.md`, with focus on the ring-based replacement for `healpy.query_disc`.

## Executive Conclusion

- I agree that the highest-value remaining target is still the HEALPix pixel-neighbor construction.
- I implemented a Numba RING-scheme cap enumerator as `pixel_backend='healpy_ring'`. It precomputes HEALPix ring geometry via `healpy.ringinfo`, enumerates rings crossing each cap, filters pixel centers by exact dot product, and returns the same downstream pixel package schema.
- Correctness is exact for the tested pz3 samples: `0` mismatched halos out of `5000` largest-paint no-shortcut halos; pair counts `127949` vs `127949`.
- The ring backend is not the current production default. It is faster for 50k/200k local samples and largest-paint 50k, but slower for the decisive 1M random chunk than the existing 16-worker `healpy` path.
- Current production recommendation remains plain `healpy`, compact pixel package, `single_pixel_angle_factor=0.5`, `pixel_workers=16`, and exact `padded_precomputed` HOD.


## Ring Backend Small/Stress Benchmarks

All rows use pz3 cap600, nside 1024, `max_paint_R200c_factor=5`, `pixel_batch_size=2000`, `single_pixel_angle_factor=0.5`, and pixel group precompute enabled.

| sample | healpy 16w s | ring s | ring delta | query_disc | ring halos | pairs |
| --- | --- | --- | --- | --- | --- | --- |
| random_50000 | 1.123 | 0.859 | -23.5% | 9660 | 9660 | 58141 |
| random_200000 | 4.125 | 3.507 | -15.0% | 39036 | 39036 | 233310 |
| largest_paint_50000 | 1.375 | 1.139 | -17.2% | 50000 | 50000 | 596753 |

## Production-Scale 1M Random Chunk

The 1M result is the promotion gate. The ring backend loses here, so it remains diagnostic-only.

| backend | requested workers | runtime s | query_disc | ring halos | pairs |
| --- | --- | --- | --- | --- | --- |
| healpy | 16 | 19.973 | 194990 | 0 | 1165444 |
| healpy_ring | 16 | 25.533 | 0 | 194990 | 1165444 |
| healpy | 1 | 24.118 | 194990 | 0 | 1165444 |
| healpy_ring | 1 | 24.986 | 0 | 194990 | 1165444 |
| healpy_ring_parallel | 4 | 26.911 | 0 | 194990 | 1165444 |
| healpy_ring_parallel | 8 | 25.253 | 0 | 194990 | 1165444 |
| healpy_ring_parallel | 16 | 25.169 | 0 | 194990 | 1165444 |
| healpy_ring_parallel | 32 | 25.581 | 0 | 194990 | 1165444 |

## Exact HOD Follow-Up

The compact exact-HOD backend was also audited against the prior best full-patch run. It is not a production speedup for this patch.

| run | node wall s | delta vs baseline | pixel sum s | gpu/get_sim_map sum s | galaxies |
| --- | --- | --- | --- | --- | --- |
| baseline padded_precomputed job6482911 | 427.461 | 0.0% | 820.55 | 496.14 | not in baseline summary |
| compact HOD job6484919 | 490.841 | +14.8% | 766.60 | 701.30 | 187166 |

## Audit Of Recommended Work

- `C++/pybind11 HEALPix cap-query backend`: still the most promising path for full-sky production. The Numba prototype validates the ring math, but does not beat optimized `healpy` plus multiprocessing at 1M.
- `Cython or Numba ring-based cap enumerator`: implemented as a Numba prototype and validated. Keep it available for diagnostics and as a reference for a future C++/OpenMP implementation, but do not promote it.
- `Exploit small-radius structure`: a `healpy_stencil` diagnostic backend exists, but the ring backend was prioritized after this audit. Stencil should not be production until separately validated by radius bins.
- `Chunk by angular radius`: still a good future direction, especially if combined with a C++ ring/stencil implementation.
- `GPU-side pixel assignment`: still higher risk and lower priority than a CPU C++/OpenMP cap enumerator.
- `Exact HOD optimization`: compact HOD is correct but slower in the full-patch timing; keep exact `padded_precomputed` as production default.
- `Profile table cache`: still useful for repeated runs, but it does not address the dominant full-sky scaling limit.


In [ ]:
from pathlib import Path
import json
import pandas as pd
repo = Path('/mnt/ceph/users/spandey/ltu-godmax/GODMAX')
meas = repo / 'data/xDESI/processed/abacus_backlight/stage31_pz3_cap600/measurements'
ring_small = json.loads((meas / 'ring_pixel_backend_benchmark_20260607.json').read_text())
ring_1m = json.loads((meas / 'ring_pixel_backend_benchmark_1M_20260607.json').read_text())
ring_parallel = json.loads((meas / 'ring_pixel_backend_benchmark_1M_parallel_20260607.json').read_text())
pd.DataFrame(ring_small['rows'])

In [ ]:
pd.DataFrame(ring_1m['rows'] + ring_parallel['parallel_rows'])

In [ ]:
compact = json.loads((meas / 'pz3_cap600_nside1024_4gpu_compact_hod_timing_summary_job6484919.json').read_text())
baseline = json.loads((meas / 'pz3_cap600_nside1024_4gpu_1M_no_prefetch_timing_summary_job6482911.json').read_text())
{
    'baseline_node_wall_time_s': baseline['node_wall_time_s'],
    'compact_node_wall_time_s': compact['node_wall_time_s'],
    'compact_delta_pct': compact['delta_pct'],
    'compact_total_galaxies': compact['total_galaxies'],
}
